# NBA Closing Lines Upload to Supabase

Extracts Pinnacle closing moneylines from JSON files and uploads to Supabase.

**Schema**: One row per game with Pinnacle closing odds only (sharpest market).

In [ ]:
import json
import os
from datetime import datetime
from supabase import create_client, Client
import pandas as pd

In [ ]:
# Supabase credentials
SUPABASE_URL = os.environ.get('SUPABASE_URL')  # or hardcode
SUPABASE_KEY = os.environ.get('SUPABASE_KEY')  # or hardcode

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

## SQL Schema (Run this in Supabase SQL Editor first)

```sql
CREATE TABLE IF NOT EXISTS nba_closing_lines (
    game_id SERIAL PRIMARY KEY,
    season VARCHAR(10) NOT NULL,
    match_date TIMESTAMP NOT NULL,
    home_team VARCHAR(50) NOT NULL,
    away_team VARCHAR(50) NOT NULL,
    home_score SMALLINT NOT NULL,
    away_score SMALLINT NOT NULL,
    home_ml_close DECIMAL(6,2) NOT NULL,
    away_ml_close DECIMAL(6,2) NOT NULL,
    venue VARCHAR(100),
    venue_town VARCHAR(100),
    scraped_date TIMESTAMP,
    created_at TIMESTAMP DEFAULT NOW(),
    UNIQUE(match_date, home_team, away_team)
);

CREATE INDEX IF NOT EXISTS idx_season ON nba_closing_lines(season);
CREATE INDEX IF NOT EXISTS idx_match_date ON nba_closing_lines(match_date);
CREATE INDEX IF NOT EXISTS idx_teams ON nba_closing_lines(home_team, away_team);
```

In [ ]:
def extract_pinnacle_odds(game_data):
    """Extract Pinnacle closing moneyline from home_away_market."""
    for market in game_data['home_away_market']:
        if market['bookmaker_name'] == 'Pinnacle' and market['period'] == 'FullTime':
            return float(market['1']), float(market['2'])
    return None, None


def parse_json_file(filepath):
    """Parse JSON file and extract game data with Pinnacle odds."""
    
    # Extract season from filename (e.g., 'nba_2023_24.json' -> '2023-24')
    filename = os.path.basename(filepath)
    season = filename.replace('nba_', '').replace('.json', '').replace('_', '-')
    
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    rows = []
    missing_pinnacle = 0
    
    for game in data:
        home_ml, away_ml = extract_pinnacle_odds(game)
        
        if home_ml is None:
            missing_pinnacle += 1
            continue  # Skip games without Pinnacle odds
        
        row = {
            'season': season,
            'match_date': game['match_date'],
            'home_team': game['home_team'],
            'away_team': game['away_team'],
            'home_score': int(game['home_score']),
            'away_score': int(game['away_score']),
            'home_ml_close': home_ml,
            'away_ml_close': away_ml,
            'venue': game.get('venue'),
            'venue_town': game.get('venue_town'),
            'scraped_date': game.get('scraped_date')
        }
        rows.append(row)
    
    print(f"Parsed {len(rows)} games from {season} ({missing_pinnacle} missing Pinnacle)")
    return rows

In [ ]:
def upload_to_supabase(rows, batch_size=100):
    """Upload rows to Supabase in batches with upsert (handles duplicates)."""
    
    total = len(rows)
    uploaded = 0
    errors = []
    
    for i in range(0, total, batch_size):
        batch = rows[i:i+batch_size]
        
        try:
            # Upsert to handle duplicates (updates if exists, inserts if new)
            result = supabase.table('nba_closing_lines').upsert(
                batch,
                on_conflict='match_date,home_team,away_team'
            ).execute()
            
            uploaded += len(batch)
            print(f"Uploaded {uploaded}/{total} rows")
            
        except Exception as e:
            error_msg = f"Batch {i//batch_size + 1} failed: {str(e)}"
            errors.append(error_msg)
            print(f"ERROR: {error_msg}")
    
    if errors:
        print(f"\n{len(errors)} batches failed. Review errors above.")
    else:
        print(f"\n✓ Successfully uploaded {uploaded} games")
    
    return uploaded, errors

## Upload a Single JSON File

Run this cell for each file you want to upload.

In [ ]:
# Specify the JSON file to upload
json_file = 'nba_2023_24.json'

# Parse JSON
rows = parse_json_file(json_file)

# Preview first few rows
df = pd.DataFrame(rows)
print("\nPreview:")
print(df.head())
print(f"\nTotal rows to upload: {len(rows)}")

In [ ]:
# Upload to Supabase
uploaded, errors = upload_to_supabase(rows)

## Verify Upload

In [ ]:
# Check row count in database
result = supabase.table('nba_closing_lines').select('season', count='exact').execute()
print(f"Total rows in database: {result.count}")

# Check by season
result = supabase.table('nba_closing_lines').select('season').execute()
df = pd.DataFrame(result.data)
print("\nGames by season:")
print(df['season'].value_counts().sort_index())

## Batch Upload Multiple Files (Optional)

In [ ]:
# List all JSON files in current directory
json_files = [f for f in os.listdir('.') if f.startswith('nba_') and f.endswith('.json')]
json_files.sort()

print(f"Found {len(json_files)} JSON files:")
for f in json_files:
    print(f"  - {f}")

In [ ]:
# Upload all files
for json_file in json_files:
    print(f"\n{'='*60}")
    print(f"Processing: {json_file}")
    print('='*60)
    
    rows = parse_json_file(json_file)
    uploaded, errors = upload_to_supabase(rows)
    
print("\n✓ All files processed")